# 03 · Model training and comparison

In this notebook we:
1. Load the **training data only**. The test set stays untouched until the final evaluation.
2. Build two models with `build_pipeline()`: **Logistic Regression** and **Random Forest**
3. Compare them with **5-fold cross-validation** on precision, recall, F1 and ROC-AUC
4. Tune the Random Forest with a small **grid search**
5. Save a comparison table and the two final trained pipelines

## 1. Load the training data

In [1]:
import sys
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.preprocessing import build_pipeline

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
MODELS_DIR.mkdir(exist_ok=True)

# Training data ONLY. X_test and y_test are deliberately not loaded in this notebook.
X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()   # squeeze: one-column table -> Series

print(f"X_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} features")
print(f"Diabetes rate in training data: {y_train.mean():.1%}")

X_train: 183,579 rows x 21 features
Diabetes rate in training data: 15.3%


## 2. Build the two pipelines

In [2]:
log_reg = build_pipeline(LogisticRegression(class_weight="balanced", max_iter=1000))
forest = build_pipeline(RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1))

log_reg

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'depreca

Each pipeline is **scaler, then model**. A Random Forest doesn't actually need scaled features, because it
splits on one feature at a time, so the scale doesn't matter. We keep the same pipeline for both models anyway:
it does no harm, and it keeps the code identical for every model.

`max_iter=1000` lets Logistic Regression run more steps to find its best solution. The default (100) can
stop too early on a large dataset.

### What does `class_weight="balanced"` do?

Only about 15% of people in the training data have diabetes. Without help, a model can get a good score by
mostly predicting "no diabetes", and it would miss most of the people we care about.

`class_weight="balanced"` tells the model that **mistakes on the rare class count more**. Each class gets a
weight of:

```
weight = number of rows / (2 x number of rows in that class)
```

so a missed diabetes case costs much more than a wrongly flagged healthy person (the exact weights are below).
This pushes the model to find more of the diabetes cases (higher **recall**), at the cost of more false alarms
(lower **precision**). We don't change or copy any data: only how much each mistake counts during training.

In [3]:
counts = y_train.value_counts().sort_index()
weights = len(y_train) / (2 * counts)
print(f"Weight for 'no diabetes' (0): {weights[0]:.2f}")
print(f"Weight for 'diabetes'    (1): {weights[1]:.2f}")
print(f"So each diabetes case counts {weights[1] / weights[0]:.1f} times as much as a non-diabetes case.")

Weight for 'no diabetes' (0): 0.59
Weight for 'diabetes'    (1): 3.27
So each diabetes case counts 5.5 times as much as a non-diabetes case.


## 3. Compare the models with cross-validation

### What is cross-validation?

We can't use the test set to compare models (see notebook 02), so we need another way to get a fair score.
**5-fold cross-validation** reuses the training data:

1. Split the training data into **5 equal parts** (folds).
2. Train on 4 parts and score on the 5th part, which the model hasn't seen.
3. Repeat 5 times, so every part is used as the scoring part once.
4. Report the **mean** of the 5 scores (typical performance) and the **standard deviation** (how much it varies).

A small standard deviation means the score is stable, not just lucky on one split.

`StratifiedKFold` keeps the same diabetes rate (about 15%) in every fold, just as our train/test split did.
`shuffle=True` with `random_state=42` mixes the rows the same way every time, so the results are reproducible.
Every model below uses **the same 5 folds**, so the comparison is fair.

### The four scores

| Score | Question it answers |
|---|---|
| **Precision** | Of the people the model flags as diabetic, what share really are? |
| **Recall** | Of the people who really are diabetic, what share does the model find? |
| **F1** | A single score that balances precision and recall (their harmonic mean) |
| **ROC-AUC** | How well the model *ranks* diabetic people above non-diabetic people. 0.5 = random guessing, 1.0 = perfect |

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = ["precision", "recall", "f1", "roc_auc"]


def cross_validate_model(pipeline, name):
    """Run 5-fold cross-validation and return the mean and std of each score."""
    start = time.time()
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=SCORING)
    print(f"{name}: done in {time.time() - start:.0f} s")

    row = {"model": name}
    for metric in SCORING:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    return row


results = [
    cross_validate_model(log_reg, "Logistic Regression"),
    cross_validate_model(forest, "Random Forest (default)"),
]

Logistic Regression: done in 2 s


Random Forest (default): done in 32 s


In [5]:
def show_results(rows):
    """Print the results as 'mean ± std' so they are easy to read."""
    table = pd.DataFrame(rows).set_index("model")
    readable = pd.DataFrame(index=table.index)
    for metric in SCORING:
        readable[metric] = (table[f"{metric}_mean"].map("{:.3f}".format) + " ± "
                            + table[f"{metric}_std"].map("{:.3f}".format))
    return readable


show_results(results)

,precision,recall,f1,roc_auc
model,,,,
Logistic Regression,0.320 ± 0.001,0.754 ± 0.007,0.449 ± 0.001,0.807 ± 0.001
Random Forest (default),0.392 ± 0.004,0.423 ± 0.005,0.407 ± 0.004,0.782 ± 0.001



**What the cross-validation shows:**
- **Logistic Regression ranks people better:** ROC-AUC **0.807**, compared with **0.782** for the default Random Forest.
- **They make different trade-offs.** Logistic Regression finds **75%** of diabetes cases (recall 0.754), but only
  about **1 in 3** of the people it flags really have diabetes (precision 0.320). The default forest is more careful
  (precision 0.392) but **misses more than half** of the diabetes cases (recall 0.423).
- **The default forest is probably overfitting.** Its trees grow until they memorise the training rows, so
  `class_weight` has little effect and it does worse on unseen folds. The grid search below tests this.
- **The scores are stable.** Every standard deviation is below 0.01, so the 5 folds agree closely.


## 4. Tune the Random Forest with a grid search

### What are hyperparameters?

**Hyperparameters** are settings we choose *before* training. The model doesn't learn them from the data.
A Random Forest is a group of **decision trees** that each vote, and three settings control it:

| Hyperparameter | What it means | Values we try |
|---|---|---|
| `n_estimators` | **How many trees** vote. More trees give steadier predictions but are slower to train. | 100, 200 |
| `max_depth` | **How many questions deep** each tree can go. Deep trees can memorise the training data (overfitting); shallow trees may be too simple (underfitting). | 10, 20 |
| `min_samples_leaf` | **The minimum number of people** that must end up in each final group ("leaf"). Larger values stop a tree from making rules based on just one or two people. | 1, 10 |

The default forest has **no depth limit** and `min_samples_leaf=1`, so its trees grow until they memorise the
training data. That is a likely reason for weak results.

### How the grid search works

`GridSearchCV` tries **every combination** (2 × 2 × 2 = 8), and scores each one with the same 5-fold
cross-validation. That is 8 × 5 = 40 trainings, which takes a few minutes. It picks the combination with the
best mean **ROC-AUC**, then retrains that one on all the training data.

We use ROC-AUC to choose because it measures how well the model *ranks* people by risk, without depending on
one particular decision threshold. In the pipeline, model settings are named `model__setting`. The double
underscore means "the `setting` of the step called `model`".

In [6]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20],
    "model__min_samples_leaf": [1, 10],
}

grid = GridSearchCV(
    build_pipeline(RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
    param_grid,
    cv=cv,                # the same 5 folds as above
    scoring=SCORING,      # record all four scores...
    refit="roc_auc",      # ...but pick the winner by ROC-AUC
)

start = time.time()
grid.fit(X_train, y_train)
print(f"Grid search done in {time.time() - start:.0f} s")
print("Best settings:", grid.best_params_)
print(f"Best mean ROC-AUC: {grid.best_score_:.3f}")

Grid search done in 323 s
Best settings: {'model__max_depth': 20, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}
Best mean ROC-AUC: 0.812


In [7]:
# Every combination, best first
grid_table = pd.DataFrame(grid.cv_results_)
grid_table = grid_table[["param_model__n_estimators", "param_model__max_depth", "param_model__min_samples_leaf",
                         "mean_test_roc_auc", "std_test_roc_auc", "mean_test_recall", "mean_test_precision"]]
grid_table.columns = ["n_estimators", "max_depth", "min_samples_leaf",
                      "roc_auc_mean", "roc_auc_std", "recall_mean", "precision_mean"]
grid_table.sort_values("roc_auc_mean", ascending=False).round(3).reset_index(drop=True)

,n_estimators,max_depth,min_samples_leaf,roc_auc_mean,roc_auc_std,recall_mean,precision_mean
0,200,20,10,0.812,0.001,0.743,0.328
1,100,20,10,0.812,0.001,0.743,0.328
2,200,10,10,0.811,0.001,0.768,0.318
3,200,10,1,0.811,0.001,0.763,0.319
4,100,10,10,0.811,0.001,0.768,0.318
5,100,10,1,0.811,0.001,0.763,0.320
6,200,20,1,0.799,0.001,0.577,0.369
7,100,20,1,0.798,0.001,0.574,0.367



**What the grid search shows:**
- **Limiting tree growth is what matters.** The two worst combinations (ROC-AUC about 0.80) are the only ones
  with deep trees **and** `min_samples_leaf=1`, the settings that let trees memorise single people. Every
  combination that limits depth *or* leaf size scores **0.811–0.812**.
- **More trees barely help.** 100 and 200 trees give the same ROC-AUC to three decimals. The 100-tree version
  would be half the size and twice as fast, with no loss in score.
- **Best settings:** `n_estimators=200`, `max_depth=20`, `min_samples_leaf=10`, with ROC-AUC **0.812**.


## 5. Comparison table

In [8]:
# The tuned forest's cross-validation scores come straight from the grid search (same 5 folds)
best = grid.best_index_
tuned_row = {"model": "Random Forest (tuned)"}
for metric in SCORING:
    tuned_row[f"{metric}_mean"] = grid.cv_results_[f"mean_test_{metric}"][best]
    tuned_row[f"{metric}_std"] = grid.cv_results_[f"std_test_{metric}"][best]

all_results = results + [tuned_row]

comparison = pd.DataFrame(all_results).round(4)
comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
print("Saved reports/model_comparison.csv")

show_results(all_results)

Saved reports/model_comparison.csv


,precision,recall,f1,roc_auc
model,,,,
Logistic Regression,0.320 ± 0.001,0.754 ± 0.007,0.449 ± 0.001,0.807 ± 0.001
Random Forest (default),0.392 ± 0.004,0.423 ± 0.005,0.407 ± 0.004,0.782 ± 0.001
Random Forest (tuned),0.328 ± 0.002,0.743 ± 0.005,0.456 ± 0.002,0.812 ± 0.001



**Comparison summary:**

- **The tuned Random Forest scores best**, with ROC-AUC **0.812** and F1 **0.456**. Tuning fixed the overfitting:
  recall rose from 0.42 to **0.74**.
- **Logistic Regression is very close**, with ROC-AUC **0.807** and F1 **0.449**. The gap is only **0.005**, and it
  trains in about 2 seconds (compared with 30+ for the forest), saves to a tiny file, and its coefficients are easy
  to explain.
- **Both models find about 3 in 4 diabetes cases, and about 1 in 3 of their flags is correct.** This trade-off
  comes from `class_weight="balanced"` and the default 0.5 decision threshold. The threshold can be adjusted later
  if fewer false alarms matter more.
- **These are cross-validation scores on the training data.** The honest final check, on the untouched test set,
  happens in the next notebook.


## 6. Save the final trained pipelines

In [9]:
# cross_validate trains copies of the pipeline, not the pipeline itself,
# so we train Logistic Regression once more on ALL the training data.
log_reg.fit(X_train, y_train)

# The grid search already retrained the best forest on all the training data (refit="roc_auc")
best_forest = grid.best_estimator_

# compress=3 makes the files smaller on disk
joblib.dump(log_reg, MODELS_DIR / "logistic_regression.joblib", compress=3)
joblib.dump(best_forest, MODELS_DIR / "random_forest.joblib", compress=3)

for path in sorted(MODELS_DIR.glob("*.joblib")):
    print(f"Saved models/{path.name} ({path.stat().st_size / 1e6:.1f} MB)")

Saved models/logistic_regression.joblib (0.0 MB)
Saved models/random_forest.joblib (41.0 MB)


Each saved file contains the **whole pipeline** (the fitted scaler plus the trained model). In the next notebook
we can load it with `joblib.load()` and call `.predict()` on new data, with no need to repeat the scaling by hand.

The model files are in `.gitignore`, so they aren't uploaded to GitHub. Anyone can recreate them by running this notebook.